# Session 3: A tiny image evaluator

Compare the original and revised **repotting illustrations** from Block 4 with the same criteria you used in your peer interviews. A vision model will give ratings and short explanations. Your task is to **check its feedback**, not to accept it as truth.

[Open in Google Colab](https://colab.research.google.com/github/digitalideation/prompt_h2601/blob/session3-evaluation-skeleton/notebooks/05_image_evaluator.ipynb), or open this notebook in JupyterHub. You only need the notebook and your two images. No GPU or repository clone is required. Run cells in order with **Shift+Enter**.

You will make **two potentially paid OpenRouter requests**, one for each image. Do not share your API token.

## 1. Install the client

Run this cell once in a new JupyterHub or Colab environment. We use the standard OpenAI Python client to connect to OpenRouter.

In [ ]:
%pip install -q openai

## 2. Connect to OpenRouter

Copy your key from [OpenRouter](https://openrouter.ai/settings/keys) and paste it into the **hidden** input below. This step does not send a request.

We use [Gemini 3.5 Flash](https://openrouter.ai/google/gemini-3.5-flash) as the vision evaluator. Nano Banana 2.1 is used earlier to **generate** the images; here we are only evaluating them.

In [ ]:
from openai import OpenAI
from getpass import getpass

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=getpass("OpenRouter API key: ").strip(),
)
MODEL = "google/gemini-3.5-flash"

## 3. Open your two images

Download the original and revised images as PNGs. In **Colab**, upload both via the **Files** sidebar (folder icon). In **JupyterHub**, upload them with the file browser into the same folder as the notebook.

Edit the two filenames to match your uploads, then run this cell to **display** both images. Displaying them does not call the model. JPEG files work too.

In [ ]:
from IPython.display import Image, display

BASELINE = "baseline.png"
REVISED = "revised.png"

print("Baseline:")
display(Image(filename=BASELINE, width=450))
print("Revised:")
display(Image(filename=REVISED, width=450))

## 4. Review the brief and criteria

Use the **same scenario and criteria** that you used when testing with people. The three criteria below are examples: replace them with yours, but **keep them unchanged** while comparing both images.

The model receives the brief, so it cannot replicate a participant seeing the image without knowing the intended action.

In [ ]:
BRIEF = "An illustration for beginners showing a small plant being moved into a larger pot."
SCENARIO = "The illustration appears alongside the instruction: Move the plant into a larger pot."

CRITERIA = [
    "The plant and the two pots are visible.",
    "The original pot and larger destination pot can be distinguished.",
    "The intended action is visually understandable.",
]

## 5. Prepare the evaluator

Read this function once. It sends **one image, your brief and your criteria** to OpenRouter and returns text feedback. You do not need to change this code.

The model uses our four ratings: **Not met / Partly met / Clearly met / Cannot judge**. It should give visible evidence for each rating. It cannot establish what a real viewer understood.

In [ ]:
import base64
from pathlib import Path

def evaluate_image(filename):
    path = Path(filename)
    types = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".webp": "image/webp"}
    mime = types.get(path.suffix.lower())
    if not mime:
        raise ValueError("Use a PNG, JPG or WebP image.")
    image_url = f"data:{mime};base64,{base64.b64encode(path.read_bytes()).decode()}"
    criteria_text = "\n".join(f"{i}. {item}" for i, item in enumerate(CRITERIA, 1))
    instructions = (
        f"Task: {BRIEF}\nViewing scenario: {SCENARIO}\nCriteria:\n{criteria_text}\n\n"
        "For EACH criterion, give one rating (Not met, Partly met, Clearly met, Cannot judge) "
        "and one short observation grounded in the visible image. "
        "Do not add criteria or an overall score. Do not claim to know what a human viewer thinks."
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": instructions},
            {"type": "image_url", "image_url": {"url": image_url}},
        ]}],
    )
    answer = response.choices[0].message.content
    if not answer:
        raise RuntimeError("No text feedback returned. Check the request in OpenRouter.")
    return answer

## 6. Evaluate the baseline

This cell makes **one paid model request**. Read the reasons, not just the ratings. If it fails, check your OpenRouter key and credits before rerunning.

In [ ]:
baseline_feedback = evaluate_image(BASELINE)
print("BASELINE EVALUATION\n")
print(baseline_feedback)

## 7. Evaluate the revision

This makes **one more paid model request**. Keep the model, task, scenario and criteria exactly the same as in Step 6.

In [ ]:
revised_feedback = evaluate_image(REVISED)
print("REVISED EVALUATION\n")
print(revised_feedback)

## 8. Compare with the human feedback

Return to the [evaluation worksheet](../activities/evaluation_clinic.md#6-compare-human-and-automated-feedback) and record:

- Where does the model agree or disagree with your interview ratings?
- Does it point to visible evidence, or make an assumption?
- Did the revision solve the original problem? What would you test next?

**Important:** Seeing a plant and two pots is not the same as knowing whether a beginner understands how to repot the plant. Do not treat the model's rating as a ground truth or combine all ratings into a single quality score.

The notebook deliberately does not save your API key or generate new images. Restart the runtime when you finish.